# 01 · TFLite → ONNX conversion and verification

**Goal:** take MediaPipe's face detector and face landmark model, convert them to ONNX, and prove the conversion is numerically correct. ONNX is the entry point for TensorRT (notebook 02).

**You will learn:**
1. What is inside a MediaPipe `.task` bundle and a `.tflite` model.
2. How to check for problem operators *before* converting.
3. What the converter changes (weights, layout, graph optimizations) and how to read its log.
4. Why NHWC vs NCHW layout matters for GPUs.
5. How to verify a conversion with the right error metrics and tolerances.
6. A first latency comparison: TFLite on CPU vs ONNX Runtime on CPU and GPU.

```
 .task bundle ──unzip──▶ face_detector.tflite ──tf2onnx──▶ detector.onnx  ──▶ (notebook 02: TensorRT)
                         face_landmarks_detector.tflite ──▶ landmarks.onnx
```

In [ ]:
# --- Setup: make the repo's src/ folder importable on the Colab runtime ---
# The notebook runs on Google's machine, which cannot see your local folders.
# So we clone the repo there (first run) or pull the latest changes (later runs).
REPO_URL = "https://github.com/UtharaKeerthan/incabin-tensorrt.git"  # change if your repo name differs

import os, sys, subprocess, pathlib
ROOT = pathlib.Path("/content/incabin-tensorrt")
if pathlib.Path("src/bench.py").exists():            # opened from the repo root
    ROOT = pathlib.Path(".").resolve()
elif pathlib.Path("../src/bench.py").exists():       # opened from notebooks/
    ROOT = pathlib.Path("..").resolve()
elif not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=False)

sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
(ROOT / "results").mkdir(exist_ok=True)
(ROOT / "models").mkdir(exist_ok=True)
print("repo root:", ROOT)

In [ ]:
%pip install -q onnx onnxruntime-gpu ai-edge-litert
%pip install -q --no-deps tf2onnx

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from src import models, convert, bench

MODELS = Path("models")
pd.set_option("display.max_colwidth", 80)

## 1. Download and unpack

A `.task` file is a zip archive. We unpack it recursively and record each file's size and hash. The download URL points at "latest", so the hash is how you prove later which model version your results came from.

In [ ]:
task = models.download(models.MODEL_URLS["face_landmarker.task"], MODELS / "face_landmarker.task")
blaze = models.download(models.MODEL_URLS["blaze_face_short_range.tflite"], MODELS / "blaze_face_short_range.tflite")

models.extract_task(task, MODELS / "face_landmarker")
tflites = models.find_files(MODELS, ".tflite")
pd.DataFrame([models.file_info(p) | {"path": str(p)} for p in tflites])

**Check:** does the standalone `blaze_face_short_range.tflite` have the same hash as the detector inside the bundle? If yes, the bundle simply reuses that model.

The bundle usually also contains a **blendshapes** model. Its input is landmarks, not an image, so we skip it.

Now pick the two models we need. If the file names differ in your download, edit this cell.

In [ ]:
def pick(keyword, exclude=()):
    hits = [p for p in tflites if keyword in p.name and not any(x in p.name for x in exclude)
            and "face_landmarker" in str(p)]
    assert len(hits) == 1, f"expected one match for {keyword!r}, got {hits}"
    return hits[0]

TFL = {
    "detector": pick("detector", exclude=("landmarks",)),
    "landmarks": pick("landmarks"),
}
TFL

## 2. Look inside before converting

**Concept:** a TFLite model is a list of operators. Converters map each operator to an ONNX equivalent. Three things can go wrong:
- **Custom ops** (`CUSTOM:...`): code that only exists inside MediaPipe. No converter knows them.
- **Rare built-in ops** without an ONNX mapping.
- **Quantization ops** that need special handling.

So the first step is always an inventory. `DEQUANTIZE` ops are expected here: we downloaded the *float16* models, which store weights in FP16 to halve the file size and convert them to FP32 when the model loads.

**✍️ Prediction:** which operator will be the most common in each model?

In [ ]:
inventory = {}
for key, path in TFL.items():
    ops = convert.tflite_op_counts(path)
    inventory[key] = ops
    custom = [o for o in ops if o.startswith("CUSTOM:")]
    print(f"== {key}: {sum(ops.values())} ops, {len(ops)} types",
          "| CUSTOM OPS FOUND: " + ", ".join(custom) if custom else "| no custom ops")

pd.DataFrame(inventory).fillna(0).astype(int).sort_values("landmarks", ascending=False)

In [ ]:
IO = {}
for key, path in TFL.items():
    ins, outs = convert.tflite_io(path)
    IO[key] = (ins, outs)
    print(f"== {key}")
    for d in ins:  print("  input :", d)
    for d in outs: print("  output:", d)

**Read the shapes** (they matter for every later notebook):
- Inputs are **NHWC**: (batch, height, width, channels). That's TFLite's native layout.
- The detector input is small (typically 128×128). Its outputs are raw *anchor* predictions: hundreds of candidate boxes plus a score for each. Turning them into face boxes (decoding + NMS) is **postprocessing** that we write ourselves in notebook 06.
- The landmark model's main output is a flat vector of 478 points × (x, y, z), in pixels of its input crop.

If the inventory showed custom ops, the conversion in the next step will fail on them. That's not a dead end; it's the subject of notebook 07 (graph surgery). Note the op names and continue with whichever model converts.

---
## 3. Convert to ONNX (default NHWC layout)

**Opset** is the version of the ONNX operator set. We use 17: modern enough for all ops here, and well supported by TensorRT 10 and ONNX Runtime.

In [ ]:
ONNX = {}
for key, path in TFL.items():
    out = MODELS / f"{key}_nhwc.onnx"
    ok, log = convert.tf2onnx_convert(path, out, opset=17)
    print(f"== {key}: {'OK' if ok else 'FAILED'}")
    # The optimizer summary line is the most informative part of the log
    print("\n".join(l for l in log.splitlines() if "After optimization" in l or "ERROR" in l or "Unsupported" in l))
    if ok:
        ONNX[key + "_nhwc"] = out
    else:
        print(log[-3000:])

**How to read "After optimization: …":** tf2onnx runs graph optimizations after converting. Typical entries:
- `Cast -N`: the FP16→FP32 weight conversions (the `DEQUANTIZE` ops) were computed once at conversion time and stored as FP32 constants. This is **constant folding**. The ONNX file grows (FP32 weights are twice the size), but no runtime work is wasted.
- `Transpose -N`: the converter inserts a transpose around every layout-sensitive op (TFLite is NHWC, ONNX convolutions are NCHW), then removes the ones that cancel out. What remains sits at the model's input and output.
- `Identity`, `Reshape` removals: no-op nodes deleted.

Now compare the two graphs side by side.

In [ ]:
summ = pd.DataFrame([convert.onnx_summary(p) | {"model": k} for k, p in ONNX.items()]).set_index("model")
display(summ[["size_kB", "opset", "nodes", "parameters"]])
for k, p in ONNX.items():
    print(k, dict(convert.onnx_op_counts(p).most_common()))

## 4. Layout: NHWC vs NCHW

**Concept:** an image tensor can be stored with channels last (**NHWC**, TFLite and most CPU code) or channels first (**NCHW**, ONNX's convention and the classic GPU layout).

If the model's input is declared NHWC, the ONNX graph must start with a `Transpose` to convert to NCHW for its convolutions. That costs a full read and write of the input on every inference. The `--inputs-as-nchw` flag declares the input as NCHW instead, so the transpose disappears from the graph. The caller then supplies NCHW data, which is easy to produce on the GPU during preprocessing.

Inside TensorRT, the compiler picks its own internal layouts per layer anyway (Tensor Core kernels often prefer channels-last formats). What we control here is the **interface**: what layout data arrives in, and whether a transpose node is left in the graph.

**✍️ Prediction:** how many `Transpose` nodes will remain in each version?

In [ ]:
for key, path in TFL.items():
    if key + "_nhwc" not in ONNX:
        continue
    in_name = IO[key][0][0]["name"]
    tmp = MODELS / f"{key}_nchw_tmp.onnx"
    out = MODELS / f"{key}_nchw.onnx"
    ok, log = convert.tf2onnx_convert(path, tmp, opset=17, nchw_inputs=[in_name])
    if not ok:
        print(key, "NCHW conversion failed\n", log[-2000:]); continue
    # Give the batch dimension a readable name; TensorRT profiles in notebook 02 refer to it.
    # Caution: if the graph has reshapes hard-coded to batch 1, batch > 1 will fail at runtime.
    # Notebook 03 tests this before sweeping batch sizes.
    convert.set_batch_dim(tmp, out, "batch")
    tmp.unlink()
    ONNX[key + "_nchw"] = out

rows = []
for k, p in ONNX.items():
    c = convert.onnx_op_counts(p); s = convert.onnx_summary(p)
    rows.append({"model": k, "nodes": s["nodes"], "Transpose": c.get("Transpose", 0),
                 "input": s["inputs"][0][1], "outputs": [o[1] for o in s["outputs"]]})
pd.DataFrame(rows)

| Layout | Pros | Cons |
|---|---|---|
| NHWC input (default) | Matches camera frames and OpenCV arrays; no change for the caller | Transpose node(s) in the graph; extra memory traffic each inference |
| NCHW input | Cleaner graph; matches ONNX/TensorRT convention | Caller must produce NCHW (easy on GPU, extra step on CPU) |

From here on, the **NCHW** models are the main path. The NHWC ones stay for comparison.

---
## 5. Verify the conversion

**Concept:** a correct conversion gives *nearly* the same numbers, not bit-identical ones. Reasons:
- Floating-point addition isn't associative: summing in a different order gives slightly different results, and each runtime uses different kernels.
- Runtimes fuse operations differently (for example conv + activation in one step), which changes rounding.

So we measure the error and judge it against a tolerance:

| Metric | Meaning | Healthy FP32 conversion |
|---|---|---|
| `max_abs` | worst single difference | small relative to the output's scale |
| `rel_max` | `max_abs` ÷ largest output value | below ~1e-4 |
| `cosine` | similarity of the whole output vector | 0.99999+ |

A `rel_max` around 1e-2 or larger means a real bug (wrong layout, wrong input range, a mis-converted op), not rounding.

We test with two inputs: **random noise** (exercises every value range) and a **real face** (realistic statistics).

In [ ]:
from skimage import data as skdata
from PIL import Image

img = skdata.astronaut()                        # 512x512 RGB test photo, bundled with scikit-image
FACE_BOX = (135, 25, 315, 205)                  # (x0, y0, x1, y1) around the face; adjust if needed
face = img[FACE_BOX[1]:FACE_BOX[3], FACE_BOX[0]:FACE_BOX[2]]

def prep(key, image_u8):
    """Resize to the model's input size and scale to its expected range. Returns NHWC float32."""
    _, h, w, _ = IO[key][0][0]["shape"]
    x = np.asarray(Image.fromarray(image_u8).resize((w, h), Image.BILINEAR), np.float32)
    # MediaPipe conventions: detector expects [-1, 1], landmark model expects [0, 1]
    x = x / 127.5 - 1.0 if key == "detector" else x / 255.0
    return x[None]

rng = np.random.default_rng(0)
INPUTS = {}
for key in TFL:
    shape = IO[key][0][0]["shape"]
    lo, hi = (-1, 1) if key == "detector" else (0, 1)
    INPUTS[key] = {"random": rng.uniform(lo, hi, shape).astype(np.float32),
                   "face": prep(key, face if key == "landmarks" else img)}

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(img); ax[0].add_patch(plt.Rectangle(FACE_BOX[:2], FACE_BOX[2]-FACE_BOX[0], FACE_BOX[3]-FACE_BOX[1], fill=False, ec="lime", lw=2))
ax[0].set_title("detector input: full image"); ax[1].imshow(face); ax[1].set_title("landmark input: face crop")
for a in ax: a.axis("off")

In [ ]:
to_nchw = lambda x: np.ascontiguousarray(x.transpose(0, 3, 1, 2))

tables = []
for key in TFL:
    for inp_name, x in INPUTS[key].items():
        ref = convert.run_tflite(TFL[key], [x])
        for variant in ("nhwc", "nchw"):
            mk = f"{key}_{variant}"
            if mk not in ONNX:
                continue
            xin = x if variant == "nhwc" else to_nchw(x)
            for prov in ("cpu", "cuda"):
                try:
                    sess = convert.make_ort_session(ONNX[mk], prov)
                except Exception as e:
                    print(f"skip {mk}/{prov}: {e}"); continue
                out = convert.run_onnx(sess, [xin])
                t = convert.compare_outputs(ref, out, label=f"{mk} ORT-{prov}")
                t.insert(1, "input", inp_name)
                tables.append(t)

verify = pd.concat(tables, ignore_index=True)
verify["pass (rel_max<1e-4)"] = verify["rel_max"] < 1e-4
verify.style.format({"max_abs": "{:.2e}", "mean_abs": "{:.2e}", "rel_max": "{:.2e}", "cosine": "{:.7f}"})

**Interpretation:**
- CPU results usually differ from TFLite by around 1e-6 relative: just rounding.
- CUDA results are often a bit larger (1e-5 range), because GPU kernels sum in a different order and may use different convolution algorithms.
- NHWC and NCHW should give the same numbers. If only one layout fails, the input transpose is wrong.
- Small outputs such as a single "face present" score can show a large *relative* error from a tiny absolute one. Check `max_abs` before worrying.

If anything fails the tolerance, check in this order: the input value range, the layout, then the converter log for warnings.

### Visual check

Numbers can pass and still hide a mistake (for example both runtimes being fed the same wrong input). So we also look at the result. The landmark output is reshaped to 478 × (x, y, z) in input-pixel coordinates.

In [ ]:
key = "landmarks"
x = INPUTS[key]["face"]
ref = convert.run_tflite(TFL[key], [x])
try:
    sess, prov = convert.make_ort_session(ONNX[f"{key}_nchw"], "cuda"), "CUDA"
except RuntimeError as e:
    print("CUDA unavailable, using CPU:", e)
    sess, prov = convert.make_ort_session(ONNX[f"{key}_nchw"], "cpu"), "CPU"
out = convert.run_onnx(sess, [to_nchw(x)])

# The landmark tensor is the output whose size is a multiple of 3 and largest
lm_name = max(ref, key=lambda k: ref[k].size if ref[k].size % 3 == 0 else -1)
lm_ref = ref[lm_name].reshape(-1, 3)
lm_onnx = (out[lm_name] if lm_name in out else list(out.values())[list(ref).index(lm_name)]).reshape(-1, 3)

pix_err = np.linalg.norm(lm_ref[:, :2] - lm_onnx[:, :2], axis=1)
print(f"{len(lm_ref)} landmarks | mean pixel shift {pix_err.mean():.2e} px | max {pix_err.max():.2e} px")

h, w = x.shape[1:3]
if np.abs(lm_ref[:, :2]).max() <= 1.5:          # normalized coordinates -> pixels
    lm_ref = lm_ref * [w, h, 1]; lm_onnx = lm_onnx * [w, h, 1]
plt.figure(figsize=(5, 5))
plt.imshow(x[0])
plt.scatter(lm_ref[:, 0], lm_ref[:, 1], s=4, c="red", label="TFLite")
plt.scatter(lm_onnx[:, 0], lm_onnx[:, 1], s=1, c="lime", label=f"ONNX ({prov})")
plt.xlim(0, w); plt.ylim(h, 0); plt.legend(loc="lower right"); plt.axis("off")
plt.title("Landmarks: should overlap exactly");

**Pixel shift** is the metric we'll keep using. In notebook 04 the same comparison against the FP32 reference tells you how much INT8 *actually* moves the landmarks, and from there how much the eye-aspect ratio (EAR) changes.

If the points sit on the face but slightly off its outline, that's the crop, not the conversion: MediaPipe normally rotates and scales the crop using the detector's output. We rebuild that step in notebook 06.

---
## 6. A first latency comparison

**✍️ Prediction:** the landmark model is small (a few MB of weights). Will ONNX Runtime on the T4 GPU be faster than TFLite on Colab's CPU, and by how much?

Note what each measurement includes:
- **TFLite (CPU, XNNPACK)**: inference only; the data is already in CPU memory.
- **ORT CPU**: same.
- **ORT CUDA with NumPy inputs**: every call copies the input to the GPU and the output back, plus launch overhead. That's the *naive* GPU path.

In [ ]:
key = "landmarks"
x = INPUTS[key]["face"]; xc = to_nchw(x)
it = convert.load_interpreter(TFL[key])
s_cpu = convert.make_ort_session(ONNX[f"{key}_nchw"], "cpu")
feed_cpu = {s_cpu.get_inputs()[0].name: xc}

res = [
    bench.benchmark(lambda: convert.run_tflite(it, [x]), name="TFLite CPU (XNNPACK)", iters=300, sync=None),
    bench.benchmark(lambda: s_cpu.run(None, feed_cpu), name="ORT CPU", iters=300, sync=None),
]
try:
    s_gpu = convert.make_ort_session(ONNX[f"{key}_nchw"], "cuda")
    feed_gpu = {s_gpu.get_inputs()[0].name: xc}
    res.append(bench.benchmark(lambda: s_gpu.run(None, feed_gpu), name="ORT CUDA (numpy in/out)", iters=500))
except RuntimeError as e:
    print("ORT CUDA skipped:", e)
display(bench.results_table(res, baseline="TFLite CPU (XNNPACK)")[["name", "p50_ms", "p99_ms", "jitter_p99_p50", "items_per_s", "speedup_vs_TFLite CPU (XNNPACK)"]])
bench.plot_latency(res, title="Landmark model, batch 1");

**Interpretation:** for a model this small, the GPU's raw speed barely matters at batch 1. What you're mostly measuring is overhead:
- copying input and output across PCIe (a fixed cost of tens of µs each way, from notebook 00),
- launching one GPU kernel per layer (a few µs each, dozens of layers),
- ONNX Runtime's own scheduling.

If the GPU only modestly beats the CPU, or even loses, that's a correct and important result: **a GPU earns its keep with work that's big enough or batched enough to fill it.** The next notebooks attack each overhead in turn:

| Overhead | Fixed in |
|---|---|
| Generic kernels, unfused layers | 02: TensorRT engine (fusion, tactic selection) |
| Batch-1 underuse of 40 SMs | 03: batch sweep and roofline |
| Kernel launch cost | 05: CUDA Graphs |
| Copies and CPU preprocessing | 06: GPU preprocessing, pinned memory, streams |

In [ ]:
verify.to_csv("results/01_verification.csv", index=False)
bench.results_table(res).to_csv("results/01_first_latency.csv", index=False)
print("saved results; ONNX models:", *[str(p) for p in ONNX.values()], sep="\n  ")

# Optional: keep models across Colab sessions (Drive mounting may not work from VS Code;
# if it fails, download the models/ folder or rerun this notebook, which takes a few minutes).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil
    shutil.copytree("models", "/content/drive/MyDrive/incabin-tensorrt/models", dirs_exist_ok=True)
    print("copied models/ to Drive")
except Exception as e:
    print("Drive not used:", type(e).__name__)

---
## Summary

- MediaPipe's models are standard TFLite with FP16 weights. Inventorying ops first tells you whether conversion can work.
- tf2onnx folded the FP16→FP32 weight conversion into constants and removed redundant transposes.
- Declaring the input as NCHW removes the remaining input transpose.
- Verification uses relative error, cosine similarity and pixel shift, on both random and real inputs.
- At batch 1, the naive GPU path is dominated by overhead, not compute. This sets up everything that follows.

### Conversion routes, pros and cons

| Route | Pros | Cons |
|---|---|---|
| tf2onnx from TFLite (used here) | Works from the shipped model file; no training code needed | Some ops (custom, rare) don't map; graph keeps TFLite's structure |
| onnx2tf / PINTO model zoo | Community conversions of many MediaPipe models; good fallback | Third-party; verify them the same way |
| Re-implement in PyTorch, load weights, export ONNX | Full control; can change layers and do QAT | Most work; must match the original exactly |

## Exercises
1. Convert with `--opset 13` and with `--opset 19`. Do node counts change? Does verification still pass?
2. Feed the landmark model an input in [0, 255] instead of [0, 1]. Do TFLite and ONNX still agree with each other? Do the landmarks still look right? What does this tell you about verification that only compares two runtimes?
3. Time the **NHWC** ONNX model on CUDA and compare with NCHW. Is the extra transpose visible at batch 1?
4. Move `FACE_BOX` by 30 pixels. How do the landmarks change? (This previews why the detector's crop quality matters in the full pipeline.)

**Next:** notebook 02 builds TensorRT engines from `landmarks_nchw.onnx` and `detector_nchw.onnx`.